# 06 · Compression

**Goals**
- Pruning / distillation / quantisation experiments
- Report model size, training time and inference time vs PQ

In [2]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.nn.utils import prune
from tqdm.auto import tqdm

IN_KAGGLE = os.path.exists("/kaggle/input")

if IN_KAGGLE:
    if not Path("/kaggle/working/COS711-A2").exists():
        !git clone -q https://github.com/AryanMohanlall/COS711-A2.git /kaggle/working/COS711-A2
    ROOT = Path("/kaggle/working/COS711-A2")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(ROOT))

try:
    import segmentation_models_pytorch  # noqa: F401
except ImportError:
    %pip install -q segmentation-models-pytorch

from src.data import get_data_dir, load_annotations
from src.dataset import FilamentDataset
from src.model import build_model, model_size
from src.preprocess import load_preprocessed, disk_mask
from src.train import evaluate, make_loader, predict, summarise
from src.augment import NO_AUG

DATA_DIR = get_data_dir()
ART_DIR = ROOT / "artifacts"
OUT_DIR = ROOT / "outputs" / "compression"
FIG_DIR = OUT_DIR / "figures"
PRUNED_DIR = OUT_DIR / "pruned"
CKPT_DIR = ROOT / "checkpoints"
RUNS_CSV = ART_DIR / "baseline_runs.csv"
for d in (FIG_DIR, PRUNED_DIR, CKPT_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
S, FULL = 1024, 2048
BATCH = 4
WORKERS = min(4, os.cpu_count())
VAL_FOLD = 0
SMOKE = os.environ.get("COS711_SMOKE") == "1"

print("Kaggle:", IN_KAGGLE, "| data dir:", DATA_DIR)
print("root:", ROOT)
print("device:", DEVICE)


C:\Users\Administrator\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Kaggle: False | data dir: C:\Users\Administrator\Downloads\COS711-A2-Aryan-training\data\MAGFiLO_1.0_Kaggle_2026
root: c:\Users\Administrator\Downloads\COS711-A2-Aryan-training
device: cpu


# Locate the existing baseline checkpoint

In [3]:
# Find the baseline row from this repo's own artifacts, not from any earlier runtime state.
if not RUNS_CSV.exists():
    raise FileNotFoundError(
        f"{RUNS_CSV} not found. Run notebook 04 first and ensure artifacts/baseline_runs.csv is in this repo root."
    )

runs = pd.read_csv(RUNS_CSV)
runs = runs.drop_duplicates(subset="name", keep="last")
fold0 = runs.loc[runs["fold"] == VAL_FOLD].copy()
if fold0.empty:
    raise ValueError("No fold-0 baseline run found in artifacts/baseline_runs.csv.")

best_row = fold0.sort_values("pq", ascending=False).iloc[0]
BASELINE_NAME = str(best_row["name"])
LABEL = str(best_row["label"])
BASELINE_CKPT = CKPT_DIR / f"{BASELINE_NAME}.pt"
if not BASELINE_CKPT.exists():
    kaggle_hits = sorted(Path("/kaggle/input").glob(f"**/{BASELINE_NAME}.pt")) if IN_KAGGLE else []
    repo_hits = [ROOT / "checkpoints" / f"{BASELINE_NAME}.pt", ROOT / "outputs" / "smoke" / "checkpoints" / f"{BASELINE_NAME}.pt"]
    hits = [h for h in kaggle_hits + repo_hits if h.exists()]
    if hits:
        BASELINE_CKPT = hits[0]
if not BASELINE_CKPT.exists():
    raise FileNotFoundError(f"Baseline checkpoint not found for {BASELINE_NAME}. Attach/copy the checkpoint from notebook 04.")

THRESHOLD = float(best_row["threshold"])
MIN_AREA = int(best_row["min_area"])
print(f"Baseline: {BASELINE_NAME} | label={LABEL} | threshold={THRESHOLD} | min_area={MIN_AREA}")
print(f"Baseline tuned PQ in notebook 04: {float(best_row['pq']):.4f}")
print("Checkpoint:", BASELINE_CKPT)


FileNotFoundError: c:\Users\Administrator\Downloads\COS711-A2-Aryan-training\artifacts\baseline_runs.csv not found. Run notebook 04 first and ensure artifacts/baseline_runs.csv is in this repo root.

# Recreate the same fold-0 validation setup

In [ ]:
# Recreate the validation setup using the repo's same artifacts and cache layout.
images, anns = load_annotations(DATA_DIR)
excluded_path = ART_DIR / "excluded_annotations.csv"
folds_path = ART_DIR / "folds.csv"
image_stats_path = ART_DIR / "image_stats.csv"
for p in (excluded_path, folds_path, image_stats_path):
    if not p.exists():
        raise FileNotFoundError(f"Required artifact missing: {p}")

excluded = pd.read_csv(excluded_path)
anns = anns[~anns.id.isin(excluded.ann_id)].reset_index(drop=True)
folds = pd.read_csv(folds_path)
geo_full = pd.read_csv(image_stats_path).set_index("obs")[["cx", "cy", "radius"]]
train_obs = folds.loc[folds.fold != VAL_FOLD, "obs"].tolist()
val_obs = folds.loc[folds.fold == VAL_FOLD, "obs"].tolist()

probe = f"{val_obs[0]}.npy"
if IN_KAGGLE:
    cache_hits = sorted(p.parent for p in Path("/kaggle/input").glob(f"**/{probe}"))
else:
    cache_hits = [ROOT / "cache" / f"full_{S}"]
cache_hits = [p for p in cache_hits if (p / probe).exists()]
if not cache_hits:
    raise FileNotFoundError(f"Preprocessed cache file {probe} not found. Re-run notebook 02 or attach cache/full_{S}.")
cache_folder = cache_hits[0]
if cache_folder.name != f"full_{S}":
    link = Path("/tmp/cos711_cache") / f"full_{S}"
    link.parent.mkdir(parents=True, exist_ok=True)
    if not link.exists():
        link.symlink_to(cache_folder, target_is_directory=True)
    cache_dir = link.parent
else:
    cache_dir = cache_folder.parent
print(f"Train observations: {len(train_obs)} | validation observations: {len(val_obs)}")
print("Cache folder:", cache_folder)

paths = images.drop_duplicates("obs").set_index("obs").path
sample = train_obs[::max(1, len(train_obs) // 100)]
values = []
for obs in tqdm(sample, desc="Estimating normalisation"):
    disk = tuple(geo_full.loc[obs])
    x = load_preprocessed(paths[obs], disk, S, cache_dir=cache_dir)
    cx, cy, radius = (v * S / FULL for v in disk)
    on_disk = disk_mask(x.shape, cx, cy, radius, 0.985)
    values.append(x[on_disk][::7])
values = np.concatenate(values)
centre = float(np.median(values))
NORM = (centre, float(1.4826 * np.median(np.abs(values - centre))))
print("Normalisation (centre, scale):", NORM)

val_ds = FilamentDataset(val_obs, images, anns, geo_full, label=LABEL, size=S,
                         aug=NO_AUG, norm=NORM, cache_dir=cache_dir)
val_loader = make_loader(val_ds, BATCH, train=False, workers=WORKERS)
gt_fn = lambda obs: val_ds.masks(obs, FULL)
disks = {obs: val_ds.geo[obs] for obs in val_ds.obs}


# Load the baseline and define pruning helpers

In [ ]:
def load_baseline():
    model = build_model(weights=None).to(DEVICE)
    state = torch.load(BASELINE_CKPT, map_location=DEVICE)
    model.load_state_dict(state)
    return model.eval()


def eligible_pruning_parameters(model):
    params = []
    for module in model.modules():
        if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear)):
            params.append((module, "weight"))
    if not params:
        raise RuntimeError("No Conv2d/Linear weights found for pruning.")
    return params


def calculate_sparsity(model):
    total, zeros = 0, 0
    for module, name in eligible_pruning_parameters(model):
        weight = getattr(module, name).detach()
        total += weight.numel()
        zeros += torch.count_nonzero(weight == 0).item()
    return zeros / total if total else 0.0


def evaluate_model(model):
    t0 = time.perf_counter()
    probabilities = predict(model, val_loader, device=DEVICE)
    inference_seconds = time.perf_counter() - t0
    counts = evaluate(probabilities, gt_fn, disks,
                      thresholds=(THRESHOLD,), min_areas=(MIN_AREA,), size=FULL)
    metrics = summarise(counts).iloc[0]
    return {
        "pq": float(metrics.pq),
        "sq": float(metrics.sq),
        "rq": float(metrics.rq),
        "inference_ms_per_image": 1000 * inference_seconds / max(1, len(probabilities)),
    }


baseline_model = load_baseline()
BASELINE_SIZE = model_size(baseline_model)
BASELINE_FILE_MB = BASELINE_CKPT.stat().st_size / (1024 ** 2)
base_metrics = evaluate_model(baseline_model)
print("Baseline model size:", BASELINE_SIZE)
print(f"Baseline checkpoint: {BASELINE_FILE_MB:.2f} MB")
print("Baseline validation metrics:", base_metrics)


# Run global unstructured L1 pruning

In [ ]:
PRUNING_LEVELS = (0.20, 0.40, 0.60, 0.80)
results = [{
    "method": "baseline",
    "target_sparsity": 0.0,
    "actual_sparsity": calculate_sparsity(baseline_model),
    "params_M": BASELINE_SIZE["params_M"],
    "checkpoint_MB": BASELINE_FILE_MB,
    **base_metrics,
}]

for amount in PRUNING_LEVELS:
    print(f"\nPruning {amount:.0%} of eligible weights...")
    model = load_baseline()
    prune.global_unstructured(
        eligible_pruning_parameters(model),
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    actual_sparsity = calculate_sparsity(model)

    for module, name in eligible_pruning_parameters(model):
        prune.remove(module, name)

    metrics = evaluate_model(model)
    checkpoint_path = PRUNED_DIR / f"{BASELINE_NAME}_global_l1_{int(amount * 100)}pct.pt"
    torch.save(model.state_dict(), checkpoint_path)
    checkpoint_mb = checkpoint_path.stat().st_size / (1024 ** 2)
    results.append({
        "method": "global_unstructured_L1",
        "target_sparsity": amount,
        "actual_sparsity": actual_sparsity,
        "params_M": model_size(model)["params_M"],
        "checkpoint_MB": checkpoint_mb,
        **metrics,
    })
    print(f"Actual sparsity: {actual_sparsity:.1%} | PQ: {metrics['pq']:.4f} | saved: {checkpoint_path}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df["pq_change"] = results_df["pq"] - base_metrics["pq"]
results_df["checkpoint_reduction_pct"] = 100 * (1 - results_df["checkpoint_MB"] / BASELINE_FILE_MB)
results_df.to_csv(OUT_DIR / "pruning_results.csv", index=False)
results_df.round(4)


# Compare quality and compression

In [ ]:
display(results_df[[
    "method", "target_sparsity", "actual_sparsity", "params_M",
    "checkpoint_MB", "checkpoint_reduction_pct", "pq", "sq", "rq",
    "inference_ms_per_image", "pq_change"
]].round(4))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pruned = results_df.iloc[1:]
axes[0].plot(100 * pruned.target_sparsity, pruned.pq, marker="o")
axes[0].axhline(base_metrics["pq"], linestyle="--", label="Baseline PQ")
axes[0].set(xlabel="Target weight sparsity (%)", ylabel="Validation PQ", title="Pruning vs segmentation quality")
axes[0].legend()
axes[1].plot(100 * pruned.target_sparsity, pruned.inference_ms_per_image, marker="o")
axes[1].set(xlabel="Target weight sparsity (%)", ylabel="Inference time (ms / image)", title="Pruning vs inference time")
plt.tight_layout()
plt.savefig(OUT_DIR / "pruning_comparison.png", dpi=160, bbox_inches="tight")
plt.show()
